#### Imports / setup

In [ ]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics

from mlxtend.plotting import plot_confusion_matrix
from mlxtend.plotting import plot_decision_regions

from matplotlib import pyplot as plt

import tensorflow
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import optimizers
from tensorflow.keras.utils import plot_model

print(tensorflow.__version__)


### Charger les données

In [ ]:
df = pd.read_csv('./dataset/labels.csv')
df

### Traitement de données

In [ ]:
# traiter les valeurs manquantes
df = df.dropna()
# supprimer les doublons
df = df.drop_duplicates()

df

### Création du modèle

In [ ]:
# création du modèle dog_breed_model
def create_model(input_dim, output_dim):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(output_dim, activation='softmax'))
    return model
# Préparation des données
X = pd.get_dummies(df.drop(columns=['breed']), drop_first=True).astype(np.float32)
y = df['breed']
le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
input_dim = X_train.shape[1]
output_dim = len(le.classes_)
model = create_model(input_dim, output_dim)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()



### Entraînement du modèle

In [ ]:
# entrainement du modèle

history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

### Tests du modèle

In [ ]:
# évaluation du modèle
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')
# Prédictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Enregistrement des résultats dans un fichier .h5
model.save('dog_breed_model.h5')
